In [ ]:

from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor
import librosa

import torch
device = "cuda" if torch.cuda.is_available() else "cpu"

import json
import jiwer #https://github.com/jitsi/jiwer
import time

import numpy as np

<h4> Data Loader </h4>

In [2]:
with open (r"C:\Users\Admin\Desktop\ip\Automatic Speech Recognition\audios.json", "r", encoding = "utf-8") as f:
    json_file = json.load (f)

display (json_file)

"""
    Normalização do Dataset.
    É importante que tanto o dataset como a transcrição obtida pelo modelo ASR tenham a mesma formatação por isso ambos vão passar por um normalização simples de remoção de duplos espaços e conversão 
    de todas as palavras para lower case.
"""

dataset = []

for exemplo in json_file:
    """
        JOIN reconstrói a frase deixando a mesma apenas com espaços em branco normais. 
        SPLIT ajuda o JOIN, separando todas as palavras da frase.
        LOWER é auto explicativo.
    """
    dataset.append (" ".join(str(exemplo["trans"]).split()).lower())

display (dataset)

[{'audio_id': 1,
  'audio_path': 'C:/Users/Admin/Desktop/ip/Automatic Speech Recognition/audio/audio1.wav',
  'trans': 'Boa tarde É aí da papelaria Sim Oh menina dá para me guardar dois bilhetes Dois bilhetes  Sim Que bilhetes diga-me Bilhetes lá do coiso que vai acontecer na Sexta Qual é o espetáculo diga-me É lá o que acontece lá no casino, que o meu neto é que quer ir Mas eu não sei qual é Você sabe  É o da Sexta, sei lá, o meu neto é que me pediu isto, já liguei para aqui para tantos sitíos, tenho que lá ir, tenho que lá ir e agora disseram-me porque é que não liga lá pá papelaria que eles guardam É assim, mas você para comprar, tem que vir cá pagá-los Sim, está bem mas tem que mos guardar não é  Não, tem de vir cá para eu tirar na máquina, sair da impressora o papel, para você me pagar na hora, não posso guardá-los Então e depois eu chego aí ',
  'duration': 42},
 {'audio_id': 2,
  'audio_path': 'C:/Users/Admin/Desktop/ip/Automatic Speech Recognition/audio/audio2.wav',
  'trans': 

['boa tarde é aí da papelaria sim oh menina dá para me guardar dois bilhetes dois bilhetes sim que bilhetes diga-me bilhetes lá do coiso que vai acontecer na sexta qual é o espetáculo diga-me é lá o que acontece lá no casino, que o meu neto é que quer ir mas eu não sei qual é você sabe é o da sexta, sei lá, o meu neto é que me pediu isto, já liguei para aqui para tantos sitíos, tenho que lá ir, tenho que lá ir e agora disseram-me porque é que não liga lá pá papelaria que eles guardam é assim, mas você para comprar, tem que vir cá pagá-los sim, está bem mas tem que mos guardar não é não, tem de vir cá para eu tirar na máquina, sair da impressora o papel, para você me pagar na hora, não posso guardá-los então e depois eu chego aí',
 'boa tarde oh menina, até estou nervosa que vim de lá de baixo oh menina, eu já encontrei encontrou o quê encontrei a elisa ah, então eu vou passar aqui ás relações públicas e diz está bem só um momento relações públicas boa tarde olhe, ó menino eu já encontr

<h5> Model Loader </h5>

In [3]:
MODEL_PATH = r"C:\Users\Admin\Desktop\models\ASR Models\Whisper\WhisperLv3-PT-All 4Bit"

PROCESSOR = AutoProcessor.from_pretrained (MODEL_PATH)
MODEL = AutoModelForSpeechSeq2Seq.from_pretrained (MODEL_PATH, device_map = device, dtype = torch.float16)

W0825 14:46:20.457000 7796 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

<hr>

<h3> Avaliação Sistema Versão 1 </h3>

```mermaid 
flowchart LR
    ÁUDIO[Entrada do Áudio]
    --> PROCESSOR[Tokenização do Áudio \nEncoder do modelo ASR]
    --> INFERÊNCIA[Inferência do Áudio \nDecoder do modelo ASR]
    --> EVAL[Avaliação]

In [54]:
TIME = {
    "Prefill": [],
    "Decode": []
}

MODEL_TRANS = []
WER = {
    "WER_CALC": []
}
CER = {
    "CER_CALC": []
}


for line in json_file:
    #print (line["audio_path"])
    WAV, SAMPLING_RATE = librosa.load (line["audio_path"], sr = 16000, mono = True)
    #print (WAV)

    """
        ################# Begin Prefill
    """
    torch.cuda.synchronize ()
    begin = time.time ()
    
    inputs = PROCESSOR (WAV, sampling_rate = 16000, return_tensors = "pt", truncation = False)

    torch.cuda.synchronize ()
    TIME["Prefill"].append (time.time () - begin) 

    inputs = inputs["input_features"].to (device, dtype = torch.float16) # Passar para GPU, não faz parte do Prefill

    """
        ################# End Prefill
    """

    """
        ################# Begin Decode
    """
    torch.cuda.synchronize ()
    begin = time.time ()

    outputs = MODEL.generate (inputs, return_timestamps = True, task = "transcribe", language = "pt")

    torch.cuda.synchronize ()
    TIME["Decode"].append (time.time () - begin)

    """
        ################# End Decode
    """
    trans = PROCESSOR.decode (outputs)
    trans = " ".join(str(trans).split()).lower() #Normalização tal como no dataset original

    MODEL_TRANS.append (trans)

    wer = jiwer.wer (line["trans"], trans)
    cer = jiwer.cer (line["trans"], trans)
    wil = jiwer.wil (line["trans"], trans)

    WER[line["audio_id"]] = wer
    WER["WER_CALC"].append (wer)

    CER[line["audio_id"]] = cer
    CER["CER_CALC"].append (cer)



In [55]:
"""
Resultados
"""

#print (TIME)
print (MODEL_TRANS)
#print (WER)
#print (CER)


print (f"Tempo Médio de Prefill: {np.mean (TIME['Prefill'])} segundos")
print (f"Tempo Médio de Decode: {np.mean (TIME['Decode'])} segundos")
print (f"P90 de Prefill: {np.percentile (TIME['Prefill'], 95)} segundos")
print (f"P90 de Decode: {np.percentile (TIME['Decode'], 95)} segundos")

print ("---" * 50)

print (f"Média Word Error Rate: {np.mean(WER['WER_CALC'])}")

print (f"Média Character Error Rate: {np.mean(CER['CER_CALC'])}")


["['e foi boa tarde sim sim dois bilhotes sim bilhetes lá do do coiso que vai acontecer na sexta qual é o espetáculo me diz é lá o do que acontece lá no casino que o meu neto é que quer ir mas eu não sei qual é você sabe é o da sexta sei lá o meu neto é que me pediu já liguei para aqui para dentro do sítio tenho que lá ir tenho que lá ir e agora me porque é que não liga lá para a papelaria que eles guardam oh é assim eu mas vocêsim está bem mas tem que me os guardar não é']", "['e viu boa tardee ela acabou de la ela mora no trinta e doze seu pai em são paulo não mora no trinta e dois seu pai em são paulo não mora no trinta e dois seu pai em são paulo não mora no trinta e dois seu pai em são paulo não mora no trinta e dois seu pai em são paulo não mora no trinta e dois seu pai em são paulo não mora no trinta e dois seu pai em são paulo não mora no trinta e dois seu pai em são paulo não mora no trinta e dois seu pai em são paulo não mora no trinta e dois seu pai em são paulo não mora no 

<p align = "center">
    <img src = "Screenshot 2026-08-25 163621.png">
</p>

<hr>

<h3> Avaliação Sistema Versão 2 </h3>

```mermaid
flowchart LR
    ÁUDIO[Entrada do Áudio]
    --> PROCESSOR[Tokenização do Áudio \nEncoder do modelo ASR]
    --> INFERÊNCIA[Inferência do Áudio \nDecoder do modelo ASR\nBeam Search de 5]
    --> EVAL[Avaliação]

In [57]:
TIME = {
    "Prefill": [],
    "Decode": []
}

MODEL_TRANS = []
WER = {
    "WER_CALC": []
}
CER = {
    "CER_CALC": []
}


for line in json_file:
    #print (line["audio_path"])
    WAV, SAMPLING_RATE = librosa.load (line["audio_path"], sr = 16000, mono = True)
    #print (WAV)

    """
        ################# Begin Prefill
    """
    torch.cuda.synchronize ()
    begin = time.time ()
    
    inputs = PROCESSOR (WAV, sampling_rate = 16000, return_tensors = "pt", truncation = False)

    torch.cuda.synchronize ()
    TIME["Prefill"].append (time.time () - begin) 

    inputs = inputs["input_features"].to (device, dtype = torch.float16) # Passar para GPU, não faz parte do Prefill

    """
        ################# End Prefill
    """

    """
        ################# Begin Decode
    """
    torch.cuda.synchronize ()
    begin = time.time ()

    outputs = MODEL.generate (inputs, return_timestamps = True, task = "transcribe", language = "pt", num_beams = 5)

    torch.cuda.synchronize ()
    TIME["Decode"].append (time.time () - begin)

    """
        ################# End Decode
    """
    trans = PROCESSOR.decode (outputs)
    trans = " ".join(str(trans).split()).lower() #Normalização tal como no dataset original

    MODEL_TRANS.append (trans)

    wer = jiwer.wer (line["trans"], trans)
    cer = jiwer.cer (line["trans"], trans)
    wil = jiwer.wil (line["trans"], trans)

    WER[line["audio_id"]] = wer
    WER["WER_CALC"].append (wer)

    CER[line["audio_id"]] = cer
    CER["CER_CALC"].append (cer)



In [58]:
"""
Resultados
"""

#print (TIME)
print (MODEL_TRANS)
#print (WER)
#print (CER)


print (f"Tempo Médio de Prefill: {np.mean (TIME['Prefill'])} segundos")
print (f"Tempo Médio de Decode: {np.mean (TIME['Decode'])} segundos")
print (f"P90 de Prefill: {np.percentile (TIME['Prefill'], 95)} segundos")
print (f"P90 de Decode: {np.percentile (TIME['Decode'], 95)} segundos")

print ("---" * 50)

print (f"Média Word Error Rate: {np.mean(WER['WER_CALC'])}")

print (f"Média Character Error Rate: {np.mean(CER['CER_CALC'])}")


["['e foi boa tarde sim sim dois bilhotes sim bilhetes lá do do coiso que vai acontecer na sexta qual é o espetáculo me diz é lá o do que acontece lá no casino que o meu neto é que quer ir mas eu não sei qual é você sabe é o da sexta sei lá o meu neto é que me pediu já liguei para aqui para dentro do sítio tenho que lá ir tenho que lá ir e agora disseram por que é que não liga lá para a papelaria que eles guardam oh é assim eu mas vocêcomprar tenho que vir cá los sim está bem mas tem que mas guardar não é não tenho que vir cá ponho o o isso a tirar na máquina e sair da impressora ao papel para você me pagar na hora e não posso los então e depois eu cheguei a']", "['o menino eu estou nervosa que vim do lado de baixo o menino eu já encontrei encontrei a elisa o olhe o menino eu já encontrei a elisa que ela mora aqui em baixo vocês todas as noites perguntam uma vizinha minha já me tinha dito que ela mora ali em baixo temos que ligar para ela a avisare ela é que vou denunciá la ela mora no

<p align = "center">
    <img src = "Screenshot 2026-08-25 172627.png">
</p>

<hr>

<h3> Avaliação Sistema Versão 3 </h3>

```mermaid
flowchart LR
    ÁUDIO[Entrada do Áudio]
    --> PROCESSOR[Tokenização do Áudio \nEncoder do modelo ASR]
    --> INFERÊNCIA[Inferência do Áudio \nDecoder do modelo ASR\nBeam Search de 10]
    --> EVAL[Avaliação]

In [59]:
TIME = {
    "Prefill": [],
    "Decode": []
}

MODEL_TRANS = []
WER = {
    "WER_CALC": []
}
CER = {
    "CER_CALC": []
}


for line in json_file:
    #print (line["audio_path"])
    WAV, SAMPLING_RATE = librosa.load (line["audio_path"], sr = 16000, mono = True)
    #print (WAV)

    """
        ################# Begin Prefill
    """
    torch.cuda.synchronize ()
    begin = time.time ()
    
    inputs = PROCESSOR (WAV, sampling_rate = 16000, return_tensors = "pt", truncation = False)

    torch.cuda.synchronize ()
    TIME["Prefill"].append (time.time () - begin) 

    inputs = inputs["input_features"].to (device, dtype = torch.float16) # Passar para GPU, não faz parte do Prefill

    """
        ################# End Prefill
    """

    """
        ################# Begin Decode
    """
    torch.cuda.synchronize ()
    begin = time.time ()

    outputs = MODEL.generate (inputs, return_timestamps = True, task = "transcribe", language = "pt", num_beams = 10)

    torch.cuda.synchronize ()
    TIME["Decode"].append (time.time () - begin)

    """
        ################# End Decode
    """
    trans = PROCESSOR.decode (outputs)
    trans = " ".join(str(trans).split()).lower() #Normalização tal como no dataset original

    MODEL_TRANS.append (trans)

    wer = jiwer.wer (line["trans"], trans)
    cer = jiwer.cer (line["trans"], trans)
    wil = jiwer.wil (line["trans"], trans)

    WER[line["audio_id"]] = wer
    WER["WER_CALC"].append (wer)

    CER[line["audio_id"]] = cer
    CER["CER_CALC"].append (cer)



In [60]:
"""
Resultados
"""

#print (TIME)
print (MODEL_TRANS)
#print (WER)
#print (CER)


print (f"Tempo Médio de Prefill: {np.mean (TIME['Prefill'])} segundos")
print (f"Tempo Médio de Decode: {np.mean (TIME['Decode'])} segundos")
print (f"P90 de Prefill: {np.percentile (TIME['Prefill'], 95)} segundos")
print (f"P90 de Decode: {np.percentile (TIME['Decode'], 95)} segundos")

print ("---" * 50)

print (f"Média Word Error Rate: {np.mean(WER['WER_CALC'])}")

print (f"Média Character Error Rate: {np.mean(CER['CER_CALC'])}")


["['e foi boa tarde sim sim dois bilhotes sim bilhetes lá do do coiso que vai acontecer na sexta qual é o espetáculo me diz é lá o do que acontece lá no casino que o meu neto é que quer ir mas eu não sei qual é você sabe é o da sexta sei lá o meu neto é que me pediu já liguei para aqui para dentro do sítio tenho que lá ir tenho que lá ir e agora disseram por que é que não liga lá para a papelaria que eles guardam oh é assim eu mas vocêcomprar tenho que vir cá e los sim está bem mas tem que mas guardar não é não tenho que vir cá e o o esse a tirar na máquina e sair da impressora ao papel para você me pagar na hora e não posso los então e depois eu cheguei a']", "['o menino eu estou nervosa que vim do lado de baixo o menino eu já encontrei encontrei a elisa o olhe o menino eu já encontrei a elisa que ela mora aqui em baixo vocês todas as noites perguntam uma vizinha minha já me tinha dito que ela mora ali em baixo temos que ligar para ela a avisare ela é que vou denunciá la ela mora no t

<p align = "center">
    <img src = "Screenshot 2026-08-25 172914.png">
</p>

<hr>